# Ejercicio 3.1 · Primer pipeline RAG sobre PDFs propios

Corpus: la documentación oficial de Python 3.13 en PDF. Nueve ficheros, 477 páginas: el tutorial (167), la referencia del lenguaje (217) y siete howtos del propio lenguaje (funcional, descriptores, expresiones regulares, Unicode, MRO, ordenación y anotaciones). `library.pdf` son 2.419 páginas de biblioteca estándar y se sale del rango; la API C es otro público.

El enunciado pide `text-embedding-3-small` y `gpt-4o-mini`. Esas dos llamadas se pagan. Aquí no hay API de pago: Ollama sirve en local `qwen3-embedding:0.6b` para los vectores y `llama3.2` para las respuestas. ChromaDB guarda el índice en disco. El patrón de la cadena es el de LCEL.


## Qué vas a ver

- Por qué PyPDF pega las palabras de estos PDF y cómo lo corrige pdfplumber.
- Si un fragmento de 500 caracteres se puede leer solo.
- La cadena LCEL y diez preguntas: factuales, de varios sitios, con jerga, fuera del corpus, con erratas y con una formulación rara.
- Dónde falla cada respuesta mala: recuperación, contexto o generación.
- El mismo experimento con fragmentos de 300 y de 1.000, con LlamaIndex, y con un prompt que cita página y sección.


In [1]:
import os
import random
import re
import tarfile
import urllib.request
import warnings
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", category=DeprecationWarning)

import chromadb
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader, PyPDFLoader
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

RAIZ = Path(".").resolve()
PDFS = RAIZ / "data" / "pdfs"
CHROMA_DIR = RAIZ / "chroma_db"
LLAMA_DIR = RAIZ / "llama_storage"
ARCHIVO = RAIZ / "data" / "python-3.13-docs-pdf-a4.tar.bz2"
URL_ARCHIVO = "https://docs.python.org/3.13/archives/python-3.13-docs-pdf-a4.tar.bz2"

ELEGIDOS = [
    "tutorial.pdf",
    "reference.pdf",
    "howto-functional.pdf",
    "howto-descriptor.pdf",
    "howto-regex.pdf",
    "howto-unicode.pdf",
    "howto-mro.pdf",
    "howto-sorting.pdf",
    "howto-annotations.pdf",
]

CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
K = 4
MODELO_EMBEDDING = "qwen3-embedding:0.6b"
MODELO_LLM = "llama3.2"
RECONSTRUIR = False
SEMILLA = 23
X_TOLERANCE = 1.5
PREFIJO_QUERY = (
    "Instruct: Given a question about Python, retrieve the relevant documentation passage\n"
    "Query:"
)


## 1. Reunir los PDF y mirar el texto

Los nueve ficheros viven en `data/pdfs/`. Si no están, se descarga el archivo oficial y se copian solo esos nueve.

PyPDFLoader es el cargador que pide el enunciado. En estos PDF, generados con LaTeX, el hueco entre palabras es de unos 2,5 puntos y el extractor junta todo lo que está más cerca de 3. El resultado es `torepeattheexample`. pdfplumber, con `x_tolerance=1.5`, trata ese hueco como un espacio y no parte las letras: dentro de una palabra el hueco es 0 o negativo.


In [2]:
def asegurar_pdfs() -> None:
    PDFS.mkdir(parents=True, exist_ok=True)
    if all((PDFS / nombre).exists() for nombre in ELEGIDOS):
        return
    if not ARCHIVO.exists():
        print("Descargando el archivo oficial de Python 3.13...")
        urllib.request.urlretrieve(URL_ARCHIVO, ARCHIVO)
    destino = RAIZ / "data" / "archivo"
    destino.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVO, "r:bz2") as paquete:
        paquete.extractall(destino)
    origen = next(destino.rglob("tutorial.pdf")).parent
    for nombre in ELEGIDOS:
        fichero = origen / nombre
        (PDFS / nombre).write_bytes(fichero.read_bytes())


asegurar_pdfs()
for nombre in ELEGIDOS:
    lector = PdfReader(str(PDFS / nombre))
    print(f"{len(lector.pages):4d}  {nombre}")
print("total", sum(len(PdfReader(str(PDFS / n)).pages) for n in ELEGIDOS))


 167  tutorial.pdf
 217  reference.pdf
  20  howto-functional.pdf
  21  howto-descriptor.pdf
  18  howto-regex.pdf
  12  howto-unicode.pdf
  10  howto-mro.pdf
   7  howto-sorting.pdf
   5  howto-annotations.pdf
total 477


In [3]:
muestra = PDFS / "tutorial.pdf"
indice = 12
crudo = PyPDFLoader(str(muestra)).load()
fino = PDFPlumberLoader(str(muestra), text_kwargs={"x_tolerance": X_TOLERANCE, "y_tolerance": 3}).load()

print("PyPDF, primeros 500 caracteres de la página", crudo[indice].metadata.get("page_label"))
print(crudo[indice].page_content[:500])
print()
print("pdfplumber x_tolerance=1.5, los mismos 500 caracteres")
print(fino[indice].page_content[:500])


PyPDF, primeros 500 caracteres de la página 7
CHAPTER
THREE
ANINFORMALINTRODUCTIONTOPYTHON
In the following examples, input and output are distinguished by the presence or absence of prompts (»> and …):
torepeattheexample,youmusttypeeverythingaftertheprompt,whenthepromptappears;linesthatdonotbegin
withapromptareoutputfromtheinterpreter. Notethatasecondarypromptonalinebyitselfinanexamplemeans
youmusttypeablankline;thisisusedtoendamulti-linecommand.
Many of the examples in this manual, even those entered at the interactive prompt, include com

pdfplumber x_tolerance=1.5, los mismos 500 caracteres
CHAPTER
THREE
AN INFORMAL INTRODUCTION TO PYTHON
In the following examples, input and output are distinguished by the presence or absence of prompts (»> and …):
to repeat the example, you must type everything after the prompt, when the prompt appears; lines that do not begin
with a prompt are output from the interpreter. Note that a secondary prompt on a line by itself in an example means
you mu

PyPDF deja frases ilegibles. pdfplumber con esa tolerancia devuelve prosa y código que se pueden indexar. Quedan dos límites del PDF, no del troceado: el código pierde la indentación, y los iconos de nota salen como `(cid:174)`. Esos iconos se borran al limpiar la página. El índice usa pdfplumber.


In [4]:
CID = re.compile(r"\(cid:\d+\)\s*")
CABECERA = re.compile(r"^.{8,90}, Release \d[\d.]*\s*$")
PIE = re.compile(r"^\d+(?:\.\d+)+\.?\s+\S.{0,90}?\s\d{1,4}\s*$")
CONTINUA = re.compile(r"^\((?:continues on next page|continued from previous page)\)\s*$", re.I)


def secciones_del_pdf(ruta: Path) -> tuple[list[str], list[str], str]:
    lector = PdfReader(str(ruta))
    marcas = []

    def recorrer(items) -> None:
        for item in items or []:
            if isinstance(item, list):
                recorrer(item)
                continue
            titulo = getattr(item, "title", None)
            if not titulo:
                continue
            try:
                pagina = lector.get_destination_page_number(item)
            except Exception:
                continue
            marcas.append((pagina, str(titulo)))

    recorrer(lector.outline)
    marcas.sort()
    actual = [""] * len(lector.pages)
    titulo = ""
    cursor = 0
    for i in range(len(lector.pages)):
        while cursor < len(marcas) and marcas[cursor][0] <= i:
            titulo = marcas[cursor][1]
            cursor += 1
        actual[i] = titulo
    nombre = lector.metadata.title if lector.metadata and lector.metadata.title else ruta.stem
    return actual, [str(etiqueta) for etiqueta in lector.page_labels], nombre


def limpiar_pagina(texto: str) -> str:
    lineas = []
    for linea in texto.splitlines():
        recortada = linea.strip()
        if CABECERA.match(recortada) or PIE.match(recortada) or CONTINUA.match(recortada):
            continue
        lineas.append(linea)
    texto = CID.sub("", "\n".join(lineas))
    return re.sub(r"\n{3,}", "\n\n", texto).strip()


def cargar_paginas() -> list[Document]:
    paginas = []
    for nombre in ELEGIDOS:
        ruta = PDFS / nombre
        docs = PDFPlumberLoader(str(ruta), text_kwargs={"x_tolerance": X_TOLERANCE, "y_tolerance": 3}).load()
        secciones, etiquetas, titulo = secciones_del_pdf(ruta)
        for doc, seccion, etiqueta in zip(docs, secciones, etiquetas):
            texto = limpiar_pagina(doc.page_content)
            if len(texto) < 80 or texto.lstrip().lower().startswith("contents"):
                continue
            paginas.append(
                Document(
                    page_content=texto,
                    metadata={
                        "documento": titulo,
                        "archivo": nombre,
                        "pagina": etiqueta,
                        "seccion": seccion,
                    },
                )
            )
        print(f"{nombre}: {len(docs)} páginas en el PDF")
    return paginas


paginas = cargar_paginas()
print("páginas útiles", len(paginas), "caracteres", sum(len(p.page_content) for p in paginas))


tutorial.pdf: 167 páginas en el PDF
reference.pdf: 217 páginas en el PDF
howto-functional.pdf: 20 páginas en el PDF
howto-descriptor.pdf: 21 páginas en el PDF
howto-regex.pdf: 18 páginas en el PDF
howto-unicode.pdf: 12 páginas en el PDF
howto-mro.pdf: 10 páginas en el PDF
howto-sorting.pdf: 7 páginas en el PDF
howto-annotations.pdf: 5 páginas en el PDF
páginas útiles 442 caracteres 1091593


## 2. Trocear

`RecursiveCharacterTextSplitter` con 500 caracteres y 80 de solape, que es el punto de partida del enunciado. Parte por párrafos, luego por líneas y luego por espacios. Abajo hay cuatro fragmentos al azar, siempre los mismos (`SEMILLA = 23`).

Un fragmento sirve si se entiende sin la página anterior. En la referencia del lenguaje, 500 caracteres corta a menudo a media frase: el ejemplo se lee, el sujeto está en el fragmento previo. El solape de 80 recupera el final de la frase anterior, no el párrafo entero. Por eso 500 se queda como línea de base y el bonus mide 300 y 1.000, en vez de cambiar el tamaño antes de ver las respuestas.


In [5]:
def trocear(paginas: list[Document], tamano: int, solape: int = CHUNK_OVERLAP) -> list[Document]:
    return RecursiveCharacterTextSplitter(chunk_size=tamano, chunk_overlap=solape).split_documents(paginas)


fragmentos = trocear(paginas, CHUNK_SIZE, CHUNK_OVERLAP)
largos = sorted(len(f.page_content) for f in fragmentos)
print(f"fragmentos {len(fragmentos)}  mediana {largos[len(largos) // 2]}  máximo {largos[-1]}")

random.seed(SEMILLA)
for i, fragmento in enumerate(random.sample(fragmentos, 4), 1):
    print("-" * 70)
    print(i, fragmento.metadata)
    print(fragmento.page_content)


fragmentos 2698  mediana 455  máximo 500
----------------------------------------------------------------------
1 {'documento': 'The Python Language Reference', 'archivo': 'reference.pdf', 'pagina': '41', 'seccion': 'Invoking Descriptors'}
acquire behaviors that differ from other instances of the same class.
The property() function is implemented as a data descriptor. Accordingly, instances cannot override the behavior
of a property.
----------------------------------------------------------------------
2 {'documento': 'Python Tutorial', 'archivo': 'tutorial.pdf', 'pagina': '60', 'seccion': 'Saving structured data with json'}
See also
pickle - the pickle module
Contrary to JSON, pickle is a protocol which allows the serialization of arbitrarily complex Python objects. As
such, it is specific to Python and cannot be used to communicate with applications written in other languages. It
is also insecure by default: deserializing pickle data coming from an untrusted source can execute arbit

De esos cuatro, el de las cadenas y el de `re.compile` se leen solos. El de pickle también: dice para qué sirve y por qué es inseguro. El de `property()` empieza a media frase («acquire behaviors...») y la frase útil es la siguiente: `property()` es un data descriptor y una instancia no puede sustituirlo. Con el metadato de sección (`Invoking Descriptors`) se sitúa; sin él, falta el sujeto.

500 caracteres aguanta un ejemplo o una definición corta. En la referencia parte frases. El solape de 80 arrastra el final de la frase anterior, no el párrafo. Por eso se queda como línea de base y el bonus compara 300 y 1.000.


## 3. Indexar en Chroma

`qwen3-embedding:0.6b` devuelve vectores de 1.024 dimensiones. En las consultas se le antepone la instrucción que pide el modelo; los pasajes se embeben tal cual. Sin esa instrucción, una pregunta en español contra este corpus en inglés pierde margen.

Los lotes son de 64 textos. Una sola petición con los miles de fragmentos agota el tiempo de Ollama. La colección usa distancia coseno. Si ya hay tantos vectores como fragmentos y `RECONSTRUIR` es falso, no se vuelve a calcular.


In [6]:
class EmbeddingsLocales(Embeddings):
    def __init__(self, modelo: str, tamano_lote: int = 64) -> None:
        self._interno = OllamaEmbeddings(
            model=modelo,
            keep_alive=1800,
            client_kwargs={"timeout": 300},
        )
        self.tamano_lote = tamano_lote

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectores: list[list[float]] = []
        total = len(texts)
        for inicio in range(0, total, self.tamano_lote):
            lote = texts[inicio : inicio + self.tamano_lote]
            vectores.extend(self._interno.embed_documents(lote))
            print(f"  embeddings {min(inicio + self.tamano_lote, total)}/{total}", flush=True)
        return vectores

    def embed_query(self, text: str) -> list[float]:
        return self._interno.embed_query(f"{PREFIJO_QUERY} {text}")


embeddings = EmbeddingsLocales(MODELO_EMBEDDING)


def obtener_coleccion(fragmentos: list[Document], tamano: int) -> Chroma:
    nombre = f"python_pdf_{tamano}"
    metadatos = {"hnsw:space": "cosine", "chunk_size": str(tamano)}
    cliente = chromadb.PersistentClient(path=str(CHROMA_DIR))
    if nombre in [coleccion.name for coleccion in cliente.list_collections()]:
        existente = cliente.get_collection(nombre)
        misma_talla = (existente.metadata or {}).get("chunk_size") == str(tamano)
        if not RECONSTRUIR and misma_talla and existente.count() == len(fragmentos):
            print(f"{nombre}: reutilizo {existente.count()} vectores")
            return Chroma(
                client=cliente,
                collection_name=nombre,
                embedding_function=embeddings,
            )
        cliente.delete_collection(nombre)
    print(f"{nombre}: calculando {len(fragmentos)} vectores")
    return Chroma.from_documents(
        documents=fragmentos,
        embedding=embeddings,
        client=cliente,
        collection_name=nombre,
        collection_metadata=metadatos,
    )


coleccion = obtener_coleccion(fragmentos, CHUNK_SIZE)
recuperador = coleccion.as_retriever(search_kwargs={"k": K})


python_pdf_500: reutilizo 2698 vectores


## 4. La cadena LCEL

El recuperador rellena `context`, la pregunta pasa tal cual, el prompt se los da a `llama3.2` y el parser se queda con el texto. Temperatura 0. La ventana es de 8.192 tokens: cuatro fragmentos de 500 caracteres caben, y también los de 1.000.


In [7]:
def formatear_contexto(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


plantilla = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando solo el contexto. "
    "Si el contexto no contiene la respuesta, di que no lo sabes. No inventes.\n\n"
    "Contexto:\n{context}\n\n"
    "Pregunta: {question}"
)

llm = ChatOllama(model=MODELO_LLM, temperature=0, num_ctx=8192, num_predict=400, keep_alive=1800)


def construir_cadena(recuperador, formatear=formatear_contexto, prompt=plantilla):
    return (
        {"context": recuperador | formatear, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


cadena = construir_cadena(recuperador)


## 5. Diez preguntas

El corpus está en inglés. Las preguntas van en español, con la jerga en inglés cuando es la del documento (`yield`, `data descriptor`, `C3`). El embedding es multilingüe; si una respuesta falla por el idioma, el diagnóstico lo marca como recuperación.

Hay dos preguntas cuya respuesta no está en estos PDF: `library.pdf` no se indexó, y pandas no es Python el lenguaje.


In [8]:
CONSULTAS = [
    {
        "id": "listas",
        "tipo": "factual",
        "pregunta": "¿En qué se diferencia una lista de una tupla?",
        "pistas": ["immutable", "inmutable"],
        "ausente": False,
    },
    {
        "id": "yield",
        "tipo": "factual",
        "pregunta": "¿Qué hace la sentencia yield?",
        "pistas": ["generator", "generador"],
        "ausente": False,
    },
    {
        "id": "with",
        "tipo": "factual",
        "pregunta": "¿Cómo garantiza la sentencia with que un fichero se cierre?",
        "pistas": ["close", "cier"],
        "ausente": False,
    },
    {
        "id": "generadores",
        "tipo": "varios sitios",
        "pregunta": "¿En qué se diferencian una list comprehension y una generator expression, y cuándo conviene el generador?",
        "pistas": ["generator", "generador"],
        "ausente": False,
    },
    {
        "id": "limpieza",
        "tipo": "varios sitios",
        "pregunta": "¿Cómo garantizan la limpieza de recursos try/finally y la sentencia with?",
        "pistas": ["finally"],
        "ausente": False,
    },
    {
        "id": "descriptor",
        "tipo": "jerga",
        "pregunta": "¿Qué diferencia hay entre un data descriptor y un non-data descriptor?",
        "pistas": ["__set__", "data descriptor"],
        "ausente": False,
    },
    {
        "id": "mro",
        "tipo": "jerga",
        "pregunta": "¿Qué es el C3 method resolution order?",
        "pistas": ["C3", "linearization", "lineal"],
        "ausente": False,
    },
    {
        "id": "pandas",
        "tipo": "ausente",
        "pregunta": "¿Qué argumentos acepta pandas.DataFrame.merge?",
        "pistas": [],
        "ausente": True,
    },
    {
        "id": "erratas",
        "tipo": "erratas",
        "pregunta": "waht does teh yeild keyword do in a generater?",
        "pistas": ["generator", "generador"],
        "ausente": False,
    },
    {
        "id": "clausura",
        "tipo": "formulación rara",
        "pregunta": "Una función definida dentro de otra que recuerda las variables de la de fuera, ¿cómo se llama y cómo funciona?",
        "pistas": ["closure", "clausura", "enclosing", "free variable"],
        "ausente": False,
    },
]

NEGATIVAS = (
    "no está",
    "no esta",
    "no lo sé",
    "no lo se",
    "no aparece",
    "no contiene",
    "no figura",
    "no se menciona",
    "no se encuentra",
    "don't know",
    "do not know",
    "not in the",
    "not mentioned",
)


def niega(respuesta: str) -> bool:
    texto = respuesta.lower()
    return any(frase in texto for frase in NEGATIVAS)


def punto_de_fallo(consulta: dict, respuesta: str, docs: list[Document]) -> str:
    if respuesta.startswith("ERROR:"):
        return "error de ejecución"
    contexto = "\n".join(doc.page_content for doc in docs).lower()
    respuesta_min = respuesta.lower()
    pistas_en_contexto = [p for p in consulta["pistas"] if p.lower() in contexto]
    pistas_en_respuesta = [p for p in consulta["pistas"] if p.lower() in respuesta_min]
    if consulta["ausente"]:
        if niega(respuesta):
            return "ninguno: dice que no está en la documentación"
        return "generación: el corpus no lo cubre y el modelo responde igual"
    if not pistas_en_contexto:
        return "recuperación: la pista no está en los fragmentos recuperados"
    if not pistas_en_respuesta:
        return "generación: el contexto tiene la pista y la respuesta no la usa"
    fragmentos_cortos = [
        doc
        for doc in docs
        if any(p.lower() in doc.page_content.lower() for p in consulta["pistas"]) and len(doc.page_content) < 220
    ]
    if fragmentos_cortos:
        return "contexto: la pista está en un fragmento muy corto"
    return "ninguno: la pista está en el contexto y en la respuesta"


def consultar(cadena, recuperador, consulta: dict) -> dict:
    try:
        docs = recuperador.invoke(consulta["pregunta"])
        respuesta = cadena.invoke(consulta["pregunta"])
    except Exception as exc:
        docs = []
        respuesta = f"ERROR: {exc}"
    return {
        "consulta": consulta,
        "respuesta": respuesta,
        "docs": docs,
        "punto": punto_de_fallo(consulta, respuesta, docs),
    }


resultados = [consultar(cadena, recuperador, consulta) for consulta in CONSULTAS]
for resultado in resultados:
    consulta = resultado["consulta"]
    print("=" * 70)
    print(f"{consulta['id']} · {consulta['tipo']} · {resultado['punto']}")
    print(consulta["pregunta"])
    print(resultado["respuesta"].strip())


listas · factual · generación: el contexto tiene la pista y la respuesta no la usa
¿En qué se diferencia una lista de una tupla?
Una lista y una tupla se diferencian en que una lista es mutable y una tupla es imutable. Además, las listas suelen contener elementos homogéneos, mientras que las tuplas suelen contener elementos heterogéneos.
yield · factual · ninguno: la pista está en el contexto y en la respuesta
¿Qué hace la sentencia yield?
La sentencia yield suspende la ejecución del generador y almacena el estado de ejecución actual, incluyendo variables locales y intentos pendientes. Cuando el generador se vuelve a iniciar con la sentencia __anext__(), se ejecuta el cuerpo del generador hasta el siguiente yield.
with · factual · ninguno: la pista está en el contexto y en la respuesta
¿Cómo garantiza la sentencia with que un fichero se cierre?
La sentencia "with" garantiza que un fichero se cierre automáticamente después de que su suite termine, incluso si se produce un error. Esto se

## 6. Diagnóstico

Los tres puntos, en este orden:

1. **Recuperación.** El fragmento que contiene la respuesta no está entre los cuatro recuperados. El tamaño del fragmento, el embedding o la formulación de la pregunta no acercaron el pasaje.
2. **Contexto.** El pasaje sí se recuperó, pero está cortado, es ambiguo o no dice el dato de forma usable.
3. **Generación.** El dato está en el contexto y el modelo lo ignora, lo contradice o, si la pregunta cae fuera del corpus, se lo inventa.

La etiqueta de arriba es una pista automática: busca unas palabras esperadas en el contexto y en la respuesta. No sustituye la lectura. La celda siguiente imprime los fragmentos de cada respuesta que no sale limpia.


In [9]:
def imprimir_diagnostico(resultados) -> None:
    for resultado in resultados:
        if resultado["punto"].startswith("ninguno"):
            continue
        consulta = resultado["consulta"]
        print("=" * 70)
        print(consulta["id"], resultado["punto"])
        print("Respuesta:", resultado["respuesta"].strip()[:500])
        if not resultado["docs"]:
            print("Sin fragmentos.")
            continue
        for i, doc in enumerate(resultado["docs"], 1):
            meta = doc.metadata
            print(f"--- fragmento {i} · {meta.get('documento')} · {meta.get('seccion')} · p. {meta.get('pagina')}")
            print(doc.page_content[:450])
            print()


imprimir_diagnostico(resultados)


listas generación: el contexto tiene la pista y la respuesta no la usa
Respuesta: Una lista y una tupla se diferencian en que una lista es mutable y una tupla es imutable. Además, las listas suelen contener elementos homogéneos, mientras que las tuplas suelen contener elementos heterogéneos.
--- fragmento 1 · Python Tutorial · Sets · p. 38
create tuples which contain mutable objects, such as lists.
Though tuples may seem similar to lists, they are often used in different situations and for different purposes. Tuples
are immutable, and usually contain a heterogeneous sequence of elements that are accessed via unpacking (see later
in this section) or indexing (or even by attribute in the case of namedtuples). Lists are mutable, and their elements
are usually homogeneous and are acces

--- fragmento 2 · Python Tutorial · Tuples and Sequences · p. 37
del later.
5.3 Tuples and Sequences
We saw that lists and strings have many common properties, such as indexing and slicing operations. They 

La etiqueta automática busca unas palabras y unas fórmulas de negativa. En esta pasada se equivoca en tres sitios, y la lectura de los fragmentos corrige el veredicto.

- **listas.** La respuesta es buena: mutable frente a inmutable, y además el tutorial dice que las listas suelen ser homogéneas y las tuplas heterogéneas. La etiqueta dice «generación» porque el modelo escribió `imutable`, sin ene, y la pista era `inmutable`.
- **yield, with, generadores, descriptor.** Correctas. La de descriptores es la más precisa: un data descriptor define `__get__` y `__set__`; un non-data descriptor solo `__get__`.
- **limpieza.** La respuesta es correcta. Los fragmentos traen el `finally` del tutorial y el `with` que cierra el fichero. La etiqueta «contexto» salta porque uno de esos fragmentos es corto, no porque falte el dato.
- **mro.** Fallo de recuperación. Los cuatro fragmentos son la historia (C3 viene de Dylan, Python 2.3 lo adopta), el índice y el cierre del howto. La sección que define el algoritmo, «The C3 Method Resolution Order», página 3, no entra en el top 4. El modelo dice que el contexto no define qué es, y con esos fragmentos tiene razón.
- **pandas.** Acierto. Dice que no lo sabe. Los fragmentos son el `merge` del algoritmo C3 y un `sorted` de nombres, no pandas. La etiqueta dice «generación» porque la negativa fue «No lo sabes» y la lista de frases buscaba «no lo sé».
- **erratas.** Fallo de generación. A pesar de `yeild` y `generater`, los fragmentos son la frase del howto («cualquier función con `yield` es un generador») y la referencia del `yield`. El modelo responde «No lo sabes».
- **clausura.** Fallo de generación. El cuarto fragmento es la referencia: un `def` dentro de una función crea una función local, y las free variables se resuelven en el ámbito que la encierra. El modelo no lo usa.

El embedding multilingüe aguanta la pregunta en español y las erratas. Quien falla, cuando falla, es `llama3.2`: o no le llega el párrafo (MRO) o le llega y lo descarta (erratas, clausura).


## Bonus 1 · Otros tamaños de fragmento

Mismas páginas, mismo solape de 80, tamaños 300 y 1.000. Cada uno tiene su colección. La pregunta es si el fallo cambia de sitio: un fragmento de 300 suele empeorar el contexto, uno de 1.000 suele meter el párrafo entero y a veces mete también ruido.


In [10]:
def coleccion_de(tamano: int):
    trozos = fragmentos if tamano == CHUNK_SIZE else trocear(paginas, tamano)
    store = obtener_coleccion(trozos, tamano)
    recuperador = store.as_retriever(search_kwargs={"k": K})
    return construir_cadena(recuperador), recuperador


cadenas_bonus = {CHUNK_SIZE: (cadena, recuperador)}
for tamano in (300, 1000):
    cadenas_bonus[tamano] = coleccion_de(tamano)

print(f"{'id':<14} {'300':<42} {'500':<42} {'1000'}")
for consulta in CONSULTAS:
    puntos = []
    for tamano in (300, 500, 1000):
        cadena_t, recuperador_t = cadenas_bonus[tamano]
        if tamano == CHUNK_SIZE:
            hecho = next(r for r in resultados if r["consulta"]["id"] == consulta["id"])
        else:
            hecho = consultar(cadena_t, recuperador_t, consulta)
        puntos.append(hecho["punto"].split(":")[0])
    print(f"{consulta['id']:<14} {puntos[0]:<42} {puntos[1]:<42} {puntos[2]}")


python_pdf_300: reutilizo 4890 vectores
python_pdf_1000: reutilizo 1372 vectores
id             300                                        500                                        1000
listas         generación                                 generación                                 ninguno
yield          generación                                 ninguno                                    ninguno
with           contexto                                   ninguno                                    ninguno
generadores    ninguno                                    ninguno                                    ninguno
limpieza       ninguno                                    contexto                                   contexto
descriptor     contexto                                   ninguno                                    ninguno
mro            ninguno                                    ninguno                                    ninguno
pandas         generación                        

No hay un tamaño único.

1.000 encaja mejor las definiciones que ocupan un párrafo: `with` y el descriptor pasan de un fallo con 300 a una respuesta limpia. 300 gana en las dos preguntas ruidosas, erratas y clausura: el fragmento pequeño es justo el párrafo de `yield` o el de las free variables, y el modelo lo usa; con 500 y 1.000 se niega. Para este corpus, 1.000 es el tamaño de partida si la pregunta está bien formulada, y 300 solo cuando se quiere cazar una frase concreta.


## Bonus 2 · El mismo corpus con LlamaIndex

LlamaIndex recibe las mismas páginas ya extraídas. Su lector por defecto también usa PyPDF y volvería a pegar las palabras. El splitter cuenta tokens, no caracteres: 128 tokens equivalen aproximadamente a los 500 caracteres de LangChain, con 20 de solape.

El índice se guarda en `llama_storage/`. El prompt pide lo mismo: responder solo con el contexto.


In [11]:
from llama_index.core import Document as DocumentoLlama
from llama_index.core import Settings, StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

Settings.embed_model = OllamaEmbedding(
    model_name=MODELO_EMBEDDING,
    base_url="http://127.0.0.1:11434",
    embed_batch_size=64,
    keep_alive=1800,
    query_instruction=PREFIJO_QUERY,
    client_kwargs={"timeout": 300},
)
Settings.llm = Ollama(
    model=MODELO_LLM,
    base_url="http://127.0.0.1:11434",
    temperature=0,
    request_timeout=180,
    context_window=8192,
    keep_alive=1800,
    additional_kwargs={"num_predict": 400},
)

if LLAMA_DIR.exists() and not RECONSTRUIR:
    indice_llama = load_index_from_storage(StorageContext.from_defaults(persist_dir=str(LLAMA_DIR)))
    print("LlamaIndex: reutilizo", LLAMA_DIR)
else:
    if LLAMA_DIR.exists() and RECONSTRUIR:
        import shutil
        shutil.rmtree(LLAMA_DIR)
    documentos_llama = [
        DocumentoLlama(
            text=pagina.page_content,
            metadata=pagina.metadata,
            excluded_embed_metadata_keys=list(pagina.metadata),
            excluded_llm_metadata_keys=list(pagina.metadata),
        )
        for pagina in paginas
    ]
    indice_llama = VectorStoreIndex.from_documents(
        documentos_llama,
        transformations=[SentenceSplitter(chunk_size=128, chunk_overlap=20)],
        show_progress=True,
    )
    indice_llama.storage_context.persist(persist_dir=str(LLAMA_DIR))

prompt_llama = PromptTemplate(
    "Responde a la pregunta usando solo el contexto. "
    "Si el contexto no contiene la respuesta, di que no lo sabes. No inventes.\n\n"
    "Contexto:\n{context_str}\n\n"
    "Pregunta: {query_str}\n"
    "Respuesta:"
)
motor = indice_llama.as_query_engine(similarity_top_k=K, text_qa_template=prompt_llama)
recuperador_llama = indice_llama.as_retriever(similarity_top_k=K)

print(f"{'id':<14} {'LangChain':<28} LlamaIndex")
for consulta in CONSULTAS:
    langchain_punto = next(r["punto"].split(":")[0] for r in resultados if r["consulta"]["id"] == consulta["id"])
    try:
        respuesta_llama = str(motor.query(consulta["pregunta"]))
        nodos = recuperador_llama.retrieve(consulta["pregunta"])
        docs_llama = [
            Document(page_content=nodo.get_content(), metadata=nodo.metadata or {})
            for nodo in nodos
        ]
        punto_llama = punto_de_fallo(consulta, respuesta_llama, docs_llama).split(":")[0]
    except Exception as exc:
        respuesta_llama = f"ERROR: {exc}"
        punto_llama = "error"
    print(f"{consulta['id']:<14} {langchain_punto:<28} {punto_llama}")
    print(respuesta_llama.strip()[:400])
    print()


LlamaIndex: reutilizo llama_storage
id             LangChain                    LlamaIndex
listas         generación                   generación
Una lista y una tupla se diferencian en que una lista es mutable y una tupla es imutable.

yield          ninguno                      generación
No lo sabes.

with           ninguno                      contexto
La sentencia "with" garantiza que un fichero se cierre automáticamente después de que se termine su uso, incluso si se produce un error durante su procesamiento.

generadores    ninguno                      generación
No lo sabes.

limpieza       contexto                     generación
La respuesta es que garantizan la limpieza de recursos.

descriptor     ninguno                      ninguno
Un data descriptor y un non-data descriptor se diferencian en cómo se calculan las overrides con respecto a las entradas en el diccionario de instancia del objeto.

Un data descriptor es un descriptor que define tanto __get__() como __set__(), m

LlamaIndex responde el descriptor, la lista y el `with`, y se niega en `yield`, las generator expressions, el MRO, las erratas y la clausura. Con el mismo texto y el mismo modelo, se niega más veces que la cadena LCEL.

El código es más corto: documentos, splitter, índice y `query_engine`. El precio es que `chunk_size` va en tokens (128 tokens, unos 500 caracteres) y que el prompt por defecto hay que sustituirlo para que no responda de memoria. El PDF no se puede dejar a su lector: también usa pypdf y volvería a pegar las palabras. La extracción con pdfplumber es común a los dos.


LangChain obliga a montar cargador, splitter, índice, prompt y cadena. Se ve cada pieza, que es lo que este ejercicio quiere enseñar. LlamaIndex condensa eso en índice y `query_engine`, y a cambio el tamaño del fragmento está en tokens. Los PDF de Sphinx no se pueden dejar al lector por defecto de ninguno de los dos.

## Bonus 3 · Citar documento, sección y página

Los metadatos entran en el contexto, no solo en el índice. El prompt tiene que usarlos. La sección sale del marcador del PDF que cae en esa página o antes; la página es la etiqueta impresa, no el índice interno que empieza en cero.


In [12]:
def formatear_con_citas(docs: list[Document]) -> str:
    bloques = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        seccion = meta.get("seccion") or "sin sección"
        bloques.append(
            f"[Fuente {i}] documento: {meta.get('documento')} | sección: {seccion} | página: {meta.get('pagina')}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(bloques)


plantilla_citas = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando solo el contexto. "
    "Si el contexto no contiene la respuesta, di exactamente: No está en la documentación. No inventes.\n"
    "Después de cada afirmación, cita la fuente con el formato (documento, sección, página).\n\n"
    "Contexto:\n{context}\n\n"
    "Pregunta: {question}"
)

cadena_citas = construir_cadena(recuperador, formatear_con_citas, plantilla_citas)

for consulta in CONSULTAS:
    if consulta["id"] not in {"descriptor", "pandas", "listas"}:
        continue
    print("=" * 70)
    print(consulta["id"], consulta["pregunta"])
    print(cadena_citas.invoke(consulta["pregunta"]).strip())


listas ¿En qué se diferencia una lista de una tupla?
Una lista y una tupla se diferencian en que una lista es mutable y una tupla es inmutable. Además, las listas suelen contener elementos homogéneos, mientras que las tuplas suelen contener elementos heterogéneos.
descriptor ¿Qué diferencia hay entre un data descriptor y un non-data descriptor?
Un data descriptor y un non-data descriptor se diferencian en cómo se calculan las overrides con respecto a las entradas en el diccionario de instancia.

Un data descriptor define tanto __get__() como __set__(), mientras que un non-data descriptor solo define __get__() y no define __set__().
pandas ¿Qué argumentos acepta pandas.DataFrame.merge?
No está en la documentación.


El contexto de estas tres respuestas lleva documento, sección y página en cada fuente. `llama3.2` responde el contenido de listas y descriptores y no copia la cita. En pandas sí obedece la frase exacta: «No está en la documentación».

El fallo es de generación. Los metadatos están en el prompt. Un modelo de 3B local no sostiene el formato `(documento, sección, página)` aunque se le pida una línea final que empiece por `Cita:`. Para que la cita salga sola haría falta un modelo que siga el formato, o construir la cita en código a partir de los metadatos del fragmento recuperado, sin pedirle al modelo que la copie.


## Qué se lleva el corpus

El índice citable es el del lenguaje Python 3.13: tutorial, referencia y los howtos que definen descriptores, MRO, generadores y Unicode. Una pregunta de pandas o de la biblioteca estándar tiene que acabar en «no está», porque ese texto no se indexó. El filtro implícito es el propio corpus: no hay pasajes de otra tecnología con los que el modelo pueda mezclar la respuesta.
